In [ ]:
!python --version

Python 3.12.12


In [ ]:
!nvidia-smi

Sat Feb 14 10:52:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# DIRECTORIES

In [ ]:
import os

ROOT_DIR = "/content/drive/MyDrive/ocr_vs_llm_parsing"
DATA_DIR = f"{ROOT_DIR}/data"

os.makedirs(DATA_DIR, exist_ok=True)

# REQUIREMENTS

In [ ]:
%%shell
pip install vllm==0.15.1
pip install PyYAML==6.0.3
pip install uvicorn==0.40.0
pip install langchain-milvus==0.3.3
pip install arxiv==2.4.0
pip install ipykernel==7.2.0
pip install pymupdf==1.26.7
pip install pytesseract==0.3.13
pip install docling==2.73.1

In [ ]:
!pip list

Package                                  Version
---------------------------------------- -----------------
absl-py                                  1.4.0
accelerate                               1.12.0
access                                   1.1.10.post3
affine                                   2.4.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.3
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.18.3
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
anthropic                                0.79.0
antlr4-python3-runtime         

# VLLM SERVE

In [ ]:
import subprocess
import time
from datetime import datetime
from typing import Tuple


def deploy_docling(
    model_name: str,
    target_message: str = "Application startup complete",
    max_dep_time: int = 2400,
    port: int = 8000,
    gpu_memory_utilization: float = 0.90,
    max_model_length: int = 6144,
    max_num_seqs: int = 1,
    max_num_batched_tokens: int = 1024,
    host: str = "0.0.0.0",
) -> Tuple[bool, str]:
    served_model_name = "_".join(model_name.split("/")[-2:])

    cmd1 = """pkill -f "vllm.entrypoints.openai.api_server .*--port {port}" || true""".format(
        **{
            "port": port,
        }
    )

    cmd2 = """export TOKENIZERS_PARALLELISM=true

vllm serve \
    --model {model_name} \
    --served-model-name {served_model_name} \
    --host {host} \
    --port {port} \
    --gpu-memory-utilization {gpu_memory_utilization} \
    --max-model-len {max_model_length} \
    --max-num-seqs {max_num_seqs} \
    --max-num-batched-tokens {max_num_batched_tokens} \
    --disable-log-stats \
    --revision untied \
    --enable-prefix-caching \
    --trust-remote-code > {served_model_name}_{port}.log 2>&1 &

sleep 3; tail -n 80 {served_model_name}_{port}.log""".format(
        **{
            "model_name": model_name,
            "served_model_name": served_model_name,
            "port": port,
            "gpu_memory_utilization": gpu_memory_utilization,
            "max_model_length": max_model_length,
            "max_num_seqs": max_num_seqs,
            "max_num_batched_tokens": max_num_batched_tokens,
            "host": host,
        }
    )

    t0 = datetime.now()

    subprocess.run(cmd1, shell=True)

    time.sleep(3)

    subprocess.run(cmd2, shell=True)

    while True:
        time.sleep(5)

        with open(f"{served_model_name}_{port}.log", "r") as log_file:
            log_content = log_file.read()

        # Check if the target message is in the log content
        if target_message in log_content:
            break

        t1 = datetime.now()

        diff = t1 - t0
        if diff.seconds >= max_dep_time:
            return False, served_model_name, None

    return True, served_model_name, port

In [ ]:
_, served_model_name, port = deploy_docling(
    model_name="ibm-granite/granite-docling-258M",
    max_num_seqs=4
)

# DATASET

In [ ]:
import os

PDF_DIR = f"{DATA_DIR}/pdf"
os.makedirs(PDF_DIR, exist_ok=True)

In [ ]:
import os
from self_supervised_attribution.dataset import automatic_loader
from tqdm.auto import tqdm
import requests
import arxiv

max_results = 50

queries = [
    "large language models",
    "rag",
    "retrieval augmented generation",
    "document parsing",
    "document intelligence"
]
results = set()
client = arxiv.Client()
for q in queries:
    search = arxiv.Search(
        query=q,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate,
    )
    for r in client.results(search):
        results.add(r.pdf_url)

for url in tqdm(results, desc="Downloading papers"):
    name = os.path.basename(url).replace(".pdf", "")
    response = requests.get(url)
    pdf_path = f"{PDF_DIR}/{name}.pdf"
    with open(pdf_path, "wb") as f:
        f.write(response.content)

In [ ]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    VlmConvertOptions,
    VlmPipelineOptions,
)
from docling.datamodel.vlm_engine_options import (
    ApiVlmEngineOptions,
    VlmEngineType,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.pipeline.vlm_pipeline import VlmPipeline


class CustomDoclingPdfConverter:
    def __init__(self, port: int, served_model_name: str, concurrency: int):
        self.port = port
        self.base_url = f"http://localhost:{port}"

        self.prepare_converter(served_model_name, concurrency)

    def prepare_converter(self, served_model_name: str, concurrency: int):
        vlm_options = VlmConvertOptions.from_preset(
            "granite_docling",
            engine_options=ApiVlmEngineOptions(
                runtime_type=VlmEngineType.API,  # Generic API type
                url=f"{self.base_url}/v1/chat/completions",
                params={
                    "model": served_model_name,
                    "max_tokens": 4096,
                    "skip_special_tokens": False,
                    "temperature": 0.0,
                },
                timeout=90,
                concurrency=concurrency,
            ),
        )

        pipeline_options = VlmPipelineOptions(
            vlm_options=vlm_options,
            enable_remote_services=True,
            # images_scale=0.5,
            do_ocr=False,
            do_picture_description=False,
            do_picture_classification=False,
            batch_size=concurrency,
        )

        self.doc_converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(
                    pipeline_options=pipeline_options,
                    pipeline_cls=VlmPipeline,
                )
            }
        )

    def __call__(self, pdf_path: str) -> str:
        r = self.doc_converter.convert(pdf_path)
        mk = r.document.export_to_markdown(image_placeholder="")

        return mk

In [ ]:
# from self_supervised_attribution.docling.pdf_converter import CustomDoclingPdfConverter

pdf_converter = CustomDoclingPdfConverter(
    port=port,
    served_model_name=served_model_name,
    concurrency=4
)

In [ ]:
import glob
from tqdm.auto import tqdm

pdf_paths = glob.glob(f"{PDF_DIR}/*.pdf")
TXT_DIR = f"{DATA_DIR}/txt"
os.makedirs(TXT_DIR, exist_ok=True)
for path in tqdm(pdf_paths, desc="Converting PDFs"):
    filename = os.path.basename(path).replace(".pdf", "")
    out_path = f"{TXT_DIR}/{filename}.txt"
    if os.path.exists(out_path):
        continue
    try:
        mk = pdf_converter(path)
    except Exception as e:
        print(e)
        continue
    with open(out_path, "w") as f:
        f.write(mk)

Converting PDFs:   0%|          | 0/194 [00:00<?, ?it/s]

Pipeline VlmPipeline failed
